In [12]:
import pandas as pd

aspects = pd.read_csv("review_aspects.csv")


In [14]:
# Ensure sentiment labels are consistently formatted.
aspects["aspect_sentiment"] = (
    aspects["aspect_sentiment"].astype("string").str.lower().str.strip()
)

In [41]:
# Remove rows without essential identifiers or valid sentiments
aspects = aspects.dropna(
    subset=["review_row_id", "aspect", "aspect_sentiment"]
).copy()

In [42]:
valid_sentiments = {"negative", "neutral", "positive"}
aspects = aspects[
    aspects["aspect_sentiment"].isin(valid_sentiments)
].copy()

In [43]:
# --------------------------------------------------
# 1. Collapse to one record per review and aspect
# --------------------------------------------------

def summarise_review_sentiment(sentiments):
    sentiment_set = set(sentiments)

    # Conservative rule: any negative evidence takes precedence
    if "negative" in sentiment_set:
        return "negative"
    elif "positive" in sentiment_set:
        return "positive"
    else:
        return "neutral"

In [ ]:
# Preserve product identifiers and title alongside each review
review_aspect = (
    aspects.groupby(
        ["review_row_id", "parent_asin", "product_title", "aspect"],
        dropna=False
    )
    .agg(
        aspect_sentiment=("aspect_sentiment", summarise_review_sentiment),
        evidence_sentence=("evidence_sentence", "first"),
    )
    .reset_index()
)

# Each row now represents one review's sentiment for one aspect.
# Repeated mentions within that review count only once.

In [45]:
# 1. Overall summary for every aspect
aspect_summary = (
    review_aspect.groupby("aspect")
    .agg(
        total_reviews=("review_row_id", "nunique"),
        unique_products=("parent_asin", "nunique"),
        negative_reviews=(
            "aspect_sentiment",
            lambda s: s.eq("negative").sum()
        ),
        neutral_reviews=(
            "aspect_sentiment",
            lambda s: s.eq("neutral").sum()
        ),
        positive_reviews=(
            "aspect_sentiment",
            lambda s: s.eq("positive").sum()
        ),
    )
    .reset_index()
)

aspect_summary["negative_rate"] = (
    aspect_summary["negative_reviews"]
    / aspect_summary["total_reviews"]
)

aspect_summary["positive_rate"] = (
    aspect_summary["positive_reviews"]
    / aspect_summary["total_reviews"]
)

aspect_summary["neutral_rate"] = (
    aspect_summary["neutral_reviews"]
    / aspect_summary["total_reviews"]
)

In [46]:
aspect_summary = aspect_summary.sort_values(
    ["negative_rate", "total_reviews"],
    ascending=[False, False]
)


In [47]:
aspect_summary.head()

,aspect,total_reviews,unique_products,negative_reviews,neutral_reviews,positive_reviews,negative_rate,positive_rate,neutral_rate
0,Broken or missing parts,8427,7316,6971,1199,257,0.827222,0.030497,0.142281
7,Safety,1330,1269,930,250,150,0.699248,0.112782,0.187970
5,Packaging,5410,5035,2509,1702,1199,0.463771,0.221627,0.314603
8,Value for money,6898,6350,3077,2752,1069,0.446071,0.154972,0.398956
1,Delivery and arrival,1713,1664,695,496,522,0.405721,0.304729,0.289550


In [48]:
aspect_summary.to_csv("aspect_summary.csv", index=False)

print("Overall aspect summary:")
print(aspect_summary)


Overall aspect summary:
                         aspect  total_reviews  unique_products  \
0       Broken or missing parts           8427             7316   
7                        Safety           1330             1269   
5                     Packaging           5410             5035   
8               Value for money           6898             6350   
1          Delivery and arrival           1713             1664   
6        Quality and durability          29691            23052   
3  Ease of use and instructions           6204             5401   
2         Design and appearance          17493            14223   
4            Enjoyment and play          27914            20269   

   negative_reviews  neutral_reviews  positive_reviews  negative_rate  \
0              6971             1199               257       0.827222   
7               930              250               150       0.699248   
5              2509             1702              1199       0.463771   
8            

Product-level metrics for each aspect

In [49]:
product_aspect = (
    review_aspect.groupby(
        ["parent_asin", "product_title", "aspect"],
        dropna=False
    )
    .agg(
        total_reviews=("review_row_id", "nunique"),
        negative_reviews=(
            "aspect_sentiment",
            lambda s: s.eq("negative").sum()
        ),
        neutral_reviews=(
            "aspect_sentiment",
            lambda s: s.eq("neutral").sum()
        ),
        positive_reviews=(
            "aspect_sentiment",
            lambda s: s.eq("positive").sum()
        ),
    )
    .reset_index()
)

product_aspect["negative_rate"] = (
    product_aspect["negative_reviews"]
    / product_aspect["total_reviews"]
)

product_aspect["positive_rate"] = (
    product_aspect["positive_reviews"]
    / product_aspect["total_reviews"]
)

product_aspect["neutral_rate"] = (
    product_aspect["neutral_reviews"]
    / product_aspect["total_reviews"]
)


In [51]:
# Rank products within each aspect
product_aspect = product_aspect.sort_values(
    ["aspect", "negative_rate", "total_reviews"],
    ascending=[True, False, False]
)

print("\nProduct-level results:")
print(product_aspect.head(20))


Product-level results:
      parent_asin                                      product_title  \
9373   B004TAYRQ0  Melissa & Doug Eraser and Chalk Set With 24 Ch...   
79860  B0BQ482XXK  Top Race Remote Control Dump Truck - 5-Channel...   
27810  B01LALE67U  KidKraft Ultimate Corner Wooden Play Kitchen w...   
83702  B0C6YN6RRV  NATIONAL GEOGRAPHIC Rock Tumbler Kit – Hobby E...   
17027  B00ITTI3VW  Buffalo Games - Cinque Terre - 1000 Piece Jigs...   
40328  B07K6S9BN3  Crayola 5555 Kid's First Large Washable Crayon...   
42795  B07PPHB41V  KANGAROO Manual Card Shuffler for Blackjack, U...   
64026  B09488QMWN  Force1 Scoot LED Hand Operated Drone for Kids ...   
1128   B00006IBOR                                 Crayola Chalk 12ct   
2766   B000GUEXBG           PlasmaCar The Original by PlaSmart (Red)   
3116   B000KS52D2  Electronic Arcade Claw Machine Mini Candy Priz...   
4137   B000YQFJ8S    Fun Express Real Starfish Decoration (30 Piece)   
5099   B001FA22DC  Pressman Mancala - Re

Filter out products with too little evidence

In [52]:
MIN_REVIEWS = 10

product_aspect_filtered = product_aspect[
    product_aspect["total_reviews"] >= MIN_REVIEWS
].copy()

product_aspect_filtered = product_aspect_filtered.sort_values(
    ["aspect", "negative_rate", "total_reviews"],
    ascending=[True, False, False]
)

print("\nFiltered product-aspect results:")
print(product_aspect_filtered.head(20))


Filtered product-aspect results:
      parent_asin                                      product_title  \
61757  B08W943N7B                      Trademark Poker Card Shuffler   
25297  B01C6FRASG  DJI Phantom 3 Standard Quadcopter Drone with 2...   
47834  B07XZ6CSTJ             VTech Touch and Swipe Baby Phone, Blue   
78896  B0BMTTTY4N                Toysmith Mini Voice Changer, (1385)   
12882  B009VE38CI  Spirograph — Deluxe Set — Art Drawing Kit — Th...   
20683  B00TLEMSVK                Hasbro Gaming Candy Land Board Game   
23234  B015OU8W3M   Colored Pencil Set with Case, 7-Inch, Pack of 72   
3658   B000REI1FU  Crayola 68-7409: Twistables Colored Pencils, 3...   
25299  B01C6FRASG  DJI Phantom 3 Standard Quadcopter Drone with 2...   
37242  B07D4RN9NH  Crayola Twistables Colored Pencil Set (50ct), ...   
50178  B082KX76ZB  NATIONAL GEOGRAPHIC Break Open 2 Geodes Scienc...   
3219   B000MRGWRQ  Melissa & Doug Jumbo Triangular Crayons - 10-P...   
4272   B00125NZWC  Crayola Min

In [55]:
product_aspect_filtered.shape

(110, 10)

In [54]:
product_aspect_filtered.to_csv(
    "product_aspect_insights.csv",
    index=False
)